#request resources + use analy environment


salloc --partition test --time 0-04:00 --mem 20gb
module load python
mamba activate analy
cd /n/home07/than157/desktop/done-large_projects/learn-better/evolm/finetune/llama-factory/
jupyter notebook --no-browser --ip=0.0.0.0 --port=8888


In [1]:
from datasets import load_dataset
import pandas as pd
import json
from tqdm import tqdm

In [2]:
#load training set of hellaswag
dataset = load_dataset("simplescaling/data_ablation_full59K", split="train")

#print dataset info
print("Dataset info:")
print(dataset)

#convert to dataframe
df = dataset.to_pandas()

print("# samples:", df.shape[0])

df.head()


Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Dataset info:
Dataset({
    features: ['solution', 'question', 'cot_type', 'source_type', 'metadata', 'cot', 'thinking_trajectories', 'attempt'],
    num_rows: 58986
})
# samples: 58986


,solution,question,cot_type,source_type,metadata,cot,thinking_trajectories,attempt
0,Since $1 \le \sqrt{1} < \sqrt{2} < \sqrt{3} < ...,The symbol $\lfloor x \rfloor$ denotes the lar...,math,qfq/openaimath/Intermediate Algebra,"{'answer': '38', 'subject': 'Intermediate Alge...",None,[\nThe problem asks for the computation of the...,Solution:\nWe want to compute the sum $S = \lf...
1,"To prove that \( BT = 2PT \), we will use geom...",Let $ABC$ be a triangle with $AB=AC$ and ...,math,AI-MO/NuminaMath-CoT/aops_forum,"{'source': 'aops_forum', 'messages': [{'conten...",None,[Let's first draw the triangle ABC and the poi...,Solution:\nConsider triangle $ABC$. We have $\...
2,None,Find the last two digits of\n\n$$\n\sum_{k=1}^...,math,GAIR/OlympicArena/Math,"{'id': 'Math_3012', 'prompt': 'You are partici...",None,[\nThe problem asks for the last two digits of...,Solution:\nLet the given sum be $S$. We have\n...
3,### Part (1)\nTo find the condition for which ...,"Let $a,\ b$ be positive real numbers. Consid...",math,AI-MO/NuminaMath-CoT/aops_forum,"{'source': 'aops_forum', 'messages': [{'conten...",None,[The circle $C_1: (x-a)^2+y^2=a^2$ has center ...,Solution:\n(1) Let the point of tangency be $(...
4,1. **Understanding the Problem:**\n Dr. Stra...,Once in a restaurant ***Dr. Strange*** found o...,math,AI-MO/NuminaMath-CoT/aops_forum,"{'source': 'aops_forum', 'messages': [{'conten...",None,[Let the food items be numbered from 1 to 12.\...,Solution:\nLet the food items be numbered from...


In [3]:
#check relevant columns
print('Questions')
print('All unique:', df['question'].nunique() == df.shape[0])
print('# NaNs:', df['question'].isna().sum())

print('')
print('Solutions')
print('All unique:', df['solution'].nunique() == df.shape[0]) #okay if not all unique, some mulitple choice question have same response/attempt
print('# NaNs:', df['solution'].isna().sum())

print('')
print('Attempt')
print('All unique:', df['attempt'].nunique() == df.shape[0]) #okay if not all unique, some mulitple choice question have same response/attempt
print('# NaNs:', df['attempt'].isna().sum())

Questions
All unique: True
# NaNs: 0

Solutions
All unique: False
# NaNs: 4694

Attempt
All unique: False
# NaNs: 0


## process data

In [4]:
#keep only rows where attempt contains '\boxed'

#note: 
# r'\\boxed' --> prints as '\boxed'
# r'\\\\boxed' --> prints as '\\boxed'

df['attempt_contains_boxed'] = df['attempt'].str.contains(r'\\boxed')
print(df['attempt_contains_boxed'].value_counts())

#keep only rows where attempt contains '\\boxed'
df_clean = df.copy()[df['attempt_contains_boxed']]
print('\ndf_clean.shape:', df_clean.shape)

#inspect examples
i = 28
print('\nInspect example')
print(df_clean['attempt'].iloc[i])

# #inspect rows where attempt does not contain '\boxed'
# df[~df['attempt_contains_boxed']].iloc[4]['attempt']
#    #some have correct answer but wrong formatting -- still, just excluded these rows (did not bother to fix them)

attempt_contains_boxed
True     54484
False     4502
Name: count, dtype: int64

df_clean.shape: (54484, 9)

Inspect example
Solution:
Let the given sum be $S$. We have
$$ S = \log_2 \frac{2}{1} + \log_2 \frac{3}{2} + \cdots + \log_2 \frac{2009}{2008} + \log_2 \frac{2010}{2009} $$
Using the property of logarithms $\log_b x + \log_b y = \log_b (xy)$, we can write the sum as:
$$ S = \log_2 \left( \frac{2}{1} \cdot \frac{3}{2} \cdot \frac{4}{3} \cdots \frac{2009}{2008} \cdot \frac{2010}{2009} \right) $$
The product inside the logarithm is a telescoping product:
$$ \frac{2}{1} \cdot \frac{3}{2} \cdot \frac{4}{3} \cdots \frac{2009}{2008} \cdot \frac{2010}{2009} = \frac{2 \cdot 3 \cdot 4 \cdots 2009 \cdot 2010}{1 \cdot 2 \cdot 3 \cdots 2008 \cdot 2009} $$
Cancelling the common terms in the numerator and denominator, we get:
$$ \frac{2010}{1} = 2010 $$
So, the sum is $S = \log_2 2010$.

We need to find the largest integer less than $\log_2 2010$. Let this integer be $n$. Then $n < \log_2 2010$

## create json file for ft

In [5]:
### create json file

#format data for sft
data = []

for idx in tqdm(range(df_clean.shape[0])):
    row = df_clean.iloc[idx]
    item = {
        "instruction": row["question"],
        "input": "",
        "output": row["attempt"]
    }
    data.append(item)

#save to JSON file
with open("data/simplescaling.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("Final # of samples in json file:", len(data))
print("Complete!")

100%|█████████████████████████████████████████████████████████████| 54484/54484 [00:01<00:00, 32879.83it/s]


Final # of samples in json file: 54484
Complete!


In [6]:
#look at example
print(df_clean.iloc[2]['question'])
print('')
print(df_clean.iloc[2]['attempt'])

Find the last two digits of

$$
\sum_{k=1}^{2008} k\left(\begin{array}{c}
2008 \\
k
\end{array}\right)
$$

Solution:
Let the given sum be $S$. We have
$$
S = \sum_{k=1}^{2008} k\left(\begin{array}{c}
2008 \\
k
\end{array}\right)
$$
Using the identity $k \binom{n}{k} = n \binom{n-1}{k-1}$, we have
$$
S = \sum_{k=1}^{2008} 2008 \binom{2007}{k-1} = 2008 \sum_{k=1}^{2008} \binom{2007}{k-1}
$$
Let $j = k-1$. As $k$ ranges from $1$ to $2008$, $j$ ranges from $0$ to $2007$.
$$
S = 2008 \sum_{j=0}^{2007} \binom{2007}{j} = 2008 \cdot 2^{2007}
$$
We need to find the last two digits of $S$, which is $S \pmod{100}$.
$$
S = 2008 \cdot 2^{2007} \equiv 8 \cdot 2^{2007} \equiv 2^3 \cdot 2^{2007} \equiv 2^{2010} \pmod{100}
$$
We need to find $2^{2010} \pmod{100}$. We consider modulo $4$ and modulo $25$.
Modulo 4: $2^{2010} \equiv 0 \pmod{4}$ since $2010 \ge 2$.
Modulo 25: We use Euler's totient theorem. $\phi(25) = 20$.
$2010 = 100 \cdot 20 + 10$.
$2^{2010} = 2^{20 \cdot 100 + 10} = (2^{20})^{100} \cdo

# below = scratch

code will probably not run, but kept this just in case

In [ ]:
#to do
#check that all responses contain "\boxed"
#change boxed to \\boxed -- no need to do this
#check question and solution lengths -- at the end of the processing -- skipped


#things I did during processing
#keep only rows where 'attempt' is properly formatted

#### originally was trying to subset based on 'solutions' not 'attempts'
but some solutions have issues, so ultimately used 'attempt' column instead
1) not all formatted with \boxed
2) don't all have CoT (some just retrun letter or numerical answer
3) some solutions don't actually asnwer the question: eg. "questions 13 - 15 go together"

In [ ]:
#keep only rows where solution is not NaN
df_clean = df.copy()[df['solution'].notna()]

#check that solutions are unique
print('')
print('After removing NaNs')
print('Solutions are unique:', df_clean['solution'].nunique() == df_clean.shape[0]) #some solutions are letters / numbers
print('# NaNs in solutions:', df_clean['solution'].isna().sum())

In [44]:
### inspect questions/rows with repeated solutions

#find all rows where the value of 'solution' appears 2x or more
repeated_values = (
    df_clean['solution']
    .value_counts()
    .loc[lambda x: x >= 2]
    .index
)

rows_w_repeated_solutions = df_clean.copy()[df_clean['solution'].isin(repeated_values)]

#manually inspect rows with repeated solutions -- seems fine
rows_w_repeated_solutions[rows_w_repeated_solutions['solution'] == '1.8']
rows_w_repeated_solutions[rows_w_repeated_solutions['solution'] == 'A']

,solution,question,cot_type,source_type,metadata,cot,thinking_trajectories,attempt
79,A,A street comprehensive governance committee ha...,math,baber/agieval/logiqa,{'label': 0},None,"[Let the three sub-committees be A, B, and C.\...","Solution:\nLet the three sub-committees be S1,..."
125,A,"Five years ago, the hair dryer produced by the...",english,baber/agieval/lsat_lr,{'label': 0},None,[Let's break down the argument and identify th...,Let's break down the argument and identify the...
537,A,A bakery makes exactly three kinds of cookie—o...,english,baber/agieval/lsat_ar,{'label': 0},None,"[Let O be oatmeal, P be peanut butter, S be su...","Solution:\nLet O be oatmeal, P be peanut butte..."
682,A,How does genetic drift affect the hardy-weinbe...,science,OpenDFM/SciEval/biology/multiple-choice/Socrat...,"{'category': 'biology', 'topic': 'Evolution', ...",None,[1. **Analyze the question:** The question ask...,The correct answer is **C. Genetic drift cause...
687,A,In studying the autobiographies of Native Amer...,english,baber/agieval/lsat_rc,{'label': 0},None,[The question asks about the reason the author...,The correct answer is **(A) identify concepts ...
...,...,...,...,...,...,...,...,...
58575,A,There are 7 athletes participating in the men'...,math,baber/agieval/logiqa,{'label': 0},None,[Let's analyze the problem.\nThere are 7 athle...,Let's analyze each option against the given co...
58580,A,Wastewater treatment consumes a lot of electri...,math,baber/agieval/logiqa,{'label': 0},None,[Let's break down the thought process for arri...,The question asks which of the following is **...
58806,A,A 68-year-old woman is referred to the outpati...,health science,OpenDFM/SciEval/biology/multiple-choice/MedQA,"{'category': 'biology', 'topic': None, 'abilit...",None,[Here's a thinking process to arrive at the co...,The correct answer is **A. She fears not being...
58826,A,The only way economists distinguish normal pro...,math,baber/agieval/logiqa,{'label': 0},None,[Let's break down the thought process for anal...,The best answer is **(A)**. Here's why:\n\n* *...


In [ ]:
#check whether solutions that are repeated are all one word
rows_w_repeated_solutions['solution'].value_counts()


solution
B       448
D       445
C       441
A       404
E       188
       ... 
2.38      2
647       2
480       2
905       2
300       2
Name: count, Length: 329, dtype: int64

In [ ]:
word_counts = rows_w_repeated_solutions['solution'].str.split().str.len()
word_counts.value_counts()

solution
1      3388
0        70
2         5
6         4
20        4
82        2
338       2
3         2
86        2
36        2
Name: count, dtype: int64

In [ ]:
rows_w_repeated_solutions['word_count'] = (
    rows_w_repeated_solutions['solution']
    .str.split()
    .str.len()
)

rows_w_repeated_solutions[rows_w_repeated_solutions['word_count'] > 1]

In [51]:
rows_w_repeated_solutions.shape

(3481, 9)

In [ ]:
#not necessary!
# when printed in notebook, it shows as \boxed
# in the json file, it shows as \\boxed

# # Replace \boxed with \\boxed -- to match format of cot-eval (Evaluator.py, last_boxed_only_string())
# df_clean['attempt_new'] = df_clean['attempt'].str.replace(r'\\boxed', r'\\\\boxed', regex=True)

# i = 28
# print('\nInspect example')
# print(df_clean['attempt_new'].iloc[i])